In [2]:
!pip install -r requirements.txt


In [3]:
# broker_api/kite_auth.py

import os
import pyotp
import requests
from urllib.parse import urlparse, parse_qs

from kiteconnect import KiteConnect
from fastapi import HTTPException, Request

# load your .env
from dotenv import load_dotenv
load_dotenv()

API_KEY    = os.getenv("KITE_API_KEY")
API_SECRET = os.getenv("KITE_API_SECRET")
USER_ID    = os.getenv("KITE_USER_ID")
PASSWORD   = os.getenv("KITE_PASSWORD")
TOTP_KEY   = os.getenv("KITE_TOTP_KEY")


def login_headless():
    session = requests.Session()

    # 1) GET the login page (cookies)
    login_url = f"https://kite.zerodha.com/connect/login?api_key={API_KEY}&v=3"
    session.headers.update({
        "User-Agent": "Mozilla/5.0",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8"
    })
    resp = session.get(login_url)
    resp.raise_for_status()
    approve_url = resp.url

    # 2) Submit credentials
    session.headers.update({
        "Accept": "application/json, text/plain, */*",
        "Referer": login_url,
        "Origin": "https://kite.zerodha.com",
        "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8"
    })
    r1 = session.post(
        "https://kite.zerodha.com/api/login",
        data={"user_id": USER_ID, "password": PASSWORD}
    )
    if r1.status_code != 200:
        raise HTTPException(400, f"Login failed: {r1.text}")
    req_id = r1.json()["data"]["request_id"]

    # 3) Submit TOTP
    r2 = session.post(
        "https://kite.zerodha.com/api/twofa",
        data={
            "user_id": USER_ID,
            "request_id": req_id,
            "twofa_value": pyotp.TOTP(TOTP_KEY).now(),
            "twofa_type": "totp",
            "skip_session": "true",
        }
    )
    if r2.status_code != 200:
        raise HTTPException(400, f"2FA failed: {r2.text}")

    # 4) Fetch redirect with request_token
    final = session.get(approve_url + "&skip_session=true", allow_redirects=True)
    token = parse_qs(urlparse(final.url).query).get("request_token", [None])[0]
    if not token:
        raise HTTPException(400, "No request_token in final URL")

    # 5) Exchange for access_token via official library
    kite = KiteConnect(api_key=API_KEY)
    data = kite.generate_session(token, api_secret=API_SECRET)
    kite.set_access_token(data["access_token"])
    return kite, data["access_token"]


def get_kite(request: Request) -> KiteConnect:
    """
    FastAPI dependency: reads 'kite_at' cookie, returns configured KiteConnect.
    """
    token = request.cookies.get("kite_at")
    if not token:
        raise HTTPException(401, "Not authenticated; login first")
    client = KiteConnect(api_key=API_KEY)
    client.set_access_token(token)
    return client


In [4]:
# In a Jupyter cell - this is your "login endpoint"
def jupyter_login():
    """
    Simplified version of your FastAPI login endpoint for Jupyter
    """
    try:
        # Use your existing function
        kite, access_token = login_headless()
        
        # Instead of database + cookies, just return what you need
        profile = kite.profile()
        
        print("✅ Login successful!")
        print(f"Welcome {profile['user_name']}")
        
        # Return kite object and token for use in other cells
        return {
            "kite": kite,
            "access_token": access_token,
            "profile": profile
        }
        
    except Exception as e:
        print(f"❌ Login failed: {str(e)}")
        return None

# Run this cell to login:
login_result = jupyter_login()

# Now you can use the kite object:
if login_result:
    kite = login_result["kite"]
    access_token = login_result["access_token"]
    
    # Test it works
    print("Testing connection...")
    profile = kite.profile()
    print(f"Connected as: {profile['user_name']}")


# Save session to file (run after successful login)
import json

def save_session(access_token, filename="kite_session.json"):
    with open(filename, 'w') as f:
        json.dump({"access_token": access_token}, f)
    print(f"✅ Session saved to {filename}")

def load_session(filename="kite_session.json"):
    try:
        with open(filename, 'r') as f:
            data = json.load(f)
        return data["access_token"]
    except:
        return None

# After successful login:
if login_result:
    save_session(login_result["access_token"])

# At start of new notebook:
def get_kite_from_saved_session():
    access_token = load_session()
    if access_token:
        kite = KiteConnect(api_key=API_KEY)
        kite.set_access_token(access_token)
        try:
            # Test if token still valid
            kite.profile()
            print("✅ Using saved session")
            return kite
        except:
            print("❌ Saved session expired")
            return None
    return None

# Quick start in new notebooks:
kite = get_kite_from_saved_session()
if not kite:
    login_result = jupyter_login()
    kite = login_result["kite"] if login_result else None

✅ Login successful!
Welcome Kokkonda Sarada
Testing connection...
Connected as: Kokkonda Sarada
✅ Session saved to kite_session.json
✅ Using saved session


In [5]:
# In Jupyter cell - direct call (since you have 'kite' from login)
profile_data = kite.profile()

# Display:
print("👤 Profile:")
for key, value in profile_data.items():
    print(f"{key}: {value}")

👤 Profile:
user_id: YX6911
user_type: individual/res_no_nn
email: krishnatejak8@gmail.com
user_name: Kokkonda Sarada
user_shortname: Kokkonda
broker: ZERODHA
exchanges: ['NFO', 'NSE', 'BSE', 'NCO', 'MCX', 'BFO', 'MF']
products: ['CNC', 'NRML', 'MIS', 'BO', 'CO']
order_types: ['MARKET', 'LIMIT', 'SL', 'SL-M']
avatar_url: None
meta: {'demat_consent': 'physical'}


In [ ]:
# Jupyter version:
def get_orders():
    return kite.orders()


In [ ]:
# In a Jupyter cell - your "holdings endpoint"
def get_holdings():
    """
    Your FastAPI /holdings_kite endpoint converted for Jupyter
    """
    try:
        holdings_data = kite.holdings()
        return holdings_data
    except Exception as e:
        print(f"Error getting holdings: {str(e)}")
        return None

# Run it:
holdings_data = get_holdings()

# Display nicely:
if holdings_data:
    print(f"📊 Total Holdings: {len(holdings_data)}")
    for holding in holdings_data[:5]:  # Show first 5
        print(f"  {holding['tradingsymbol']}: {holding['quantity']} shares")

📊 Total Holdings: 10
  COROMANDEL: 2 shares
  ETERNAL: 180 shares
  FORTIS: 4 shares
  LAURUSLABS: 4 shares
  LTF: 15 shares


In [ ]:
# In Jupyter cell - direct call
holdings_data = kite.holdings()

# Quick analysis:
print(f"Total holdings: {len(holdings_data)}")
for h in holdings_data:
    if h['quantity'] > 0:
        print(f"{h['tradingsymbol']}: {h['quantity']} @ ₹{h['average_price']}")

Total holdings: 10
COROMANDEL: 2 @ ₹2402.4
ETERNAL: 180 @ ₹160.842857
FORTIS: 4 @ ₹943.5
LAURUSLABS: 4 @ ₹877.75
LTF: 15 @ ₹216.25
MAXHEALTH: 3 @ ₹1217.8
MFSL: 2 @ ₹1621
MUTHOOTFIN: 1 @ ₹2754
PAYTM: 3 @ ₹1202.5
TATAMOTORS: 4 @ ₹699.9


In [ ]:
# In Jupyter cell - simple order placement
def place_simple_order():
    """
    Place a basic market order
    """
    try:
        order = kite.place_order(
            variety="regular",
            exchange="NSE",
            tradingsymbol="SBIN",
            transaction_type="BUY",
            quantity=1,
            order_type="GTT",
            product="REGULAR"
        )
        print(f"Order successful! ID: {order['order_id']}")
        return order
    except Exception as e:
        print(f"Failed: {str(e)}")
        return None

# Try this first:
# result = place_simple_order()

In [ ]:
# Try this first:
result = place_simple_order()

Failed: Your order could not be converted to a After Market Order (AMO).


In [6]:
instruments_list = kite.instruments("NSE")

In [9]:
import pandas as pd
instrumentList = pd.read_csv("https://api.kite.trade/instruments")
print(instrumentList)

       instrument_token  exchange_token   tradingsymbol  \
0             215681797          842507  BANKEX25SEPFUT   
1             220537093          861473  BANKEX25OCTFUT   
2             223969541          874881  BANKEX25NOVFUT   
3             215645189          842364  SENSEX25SEPFUT   
4             220514309          861384  SENSEX25OCTFUT   
...                 ...             ...             ...   
93194         194418689          759448      SHREEJISPG   
93195         194419969          759453      VIKRAMSOLR   
93196         194421249          759458      GROWWNXT50   
93197         194422017          759461        GEMAROMA   
93198         194462721          759620    SARVES-RE-BE   

                            name  last_price      expiry  strike  tick_size  \
0                         BANKEX           0  2025-09-25     0.0       0.05   
1                         BANKEX           0  2025-10-30     0.0       0.05   
2                         BANKEX           0  2025-11-

In [ ]:
def instruments(self, exchange=None):
    """
    Retrieve the list of market instruments available to trade.
    Note that the results could be large, several hundred KBs in size,
    with tens of thousands of entries in the list.
    - `exchange` is specific exchange to fetch (Optional)
    """
    if exchange:
        return self._parse_instruments(self._get("market.instruments", url_args={"exchange": exchange}))
    else:
        return self._parse_instruments(self._get("market.instruments.all"))

NameError: name 'NSE' is not defined

In [ ]:
# Alternative method using instruments data
instruments_list = kite.instruments("NSE")
for instrument in instruments_list:
    if instrument["tradingsymbol"] == "SBIN":
        last_price = instrument["last_price"]
        print(f"Last price from instruments: {last_price}")
        break

Last price from instruments: 0.0


In [ ]:

# Try this first:
result = place_simple_order()

Failed: Your order could not be converted to a After Market Order (AMO).


In [ ]:
# Get last traded price for SBIN
try:
    ltp_data = kite.ltp(["NSE:SBIN"])  # Pass as a list
    last_price = ltp_data["NSE:SBIN"]["last_price"]
    print(f"Last price for SBIN: {last_price}")
except Exception as e:
    print(f"Error fetching LTP: {e}")

Last price for SBIN: 812.15


In [ ]:
# This should work according to the documentation
ltp_data = kite.ltp(["NSE:SBIN"])
last_price = ltp_data["NSE:SBIN"]["last_price"]
print(f"Last price for SBIN: {last_price}")

Last price for SBIN: 812.15


In [ ]:
# First get the instrument token
instruments_list = kite.instruments("NSE")
sbin_instrument_token = None
for instrument in instruments_list:
    if instrument["tradingsymbol"] == "SBIN":
        sbin_instrument_token = instrument["instrument_token"]
        break

# Then use the token to fetch LTP
if sbin_instrument_token:
    ltp_data = kite.ltp([str(sbin_instrument_token)])
    last_price = ltp_data[str(sbin_instrument_token)]["last_price"]
    print(f"Last price for SBIN: {last_price}")

Last price for SBIN: 812.15
